# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [8]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [9]:
# TODO: Import the necessary libs
# For example: 
import os

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [10]:
# Setup environment and OpenAI client
import os
from dotenv import load_dotenv
from openai import OpenAI

# Ensure the .env file is loaded and overrides existing keys
load_dotenv(override=True)

print("OPENAI_API_KEY:", os.getenv("OPENAI_API_KEY"))
print("OPENAI_BASE_URL:", os.getenv("OPENAI_BASE_URL"))

# Extra sanity check: Agent class import must be available
from lib.agents import Agent
print("Agent class available:", Agent)

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)

OPENAI_API_KEY: voc-128284763016886546721826946fbfde31796.64664478
OPENAI_BASE_URL: https://openai.vocareum.com/v1
Agent class available: <class 'lib.agents.Agent'>


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [11]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

from lib.tooling import tool
import chromadb
import os
from embedding import VocareumEmbeddingFunction


# Initialize ChromaDB client and collection
chroma_client = chromadb.PersistentClient(path="chromadb")
embedding_fn = VocareumEmbeddingFunction(os.getenv("OPENAI_API_KEY"))
collection = chroma_client.get_collection(
    name="udaplay",
    embedding_function=embedding_fn
)


# def retrieve_game(query: str):
#     """
#     Semantic search: Finds most relevant results in the vector DB.
#     Args:
#         query (str): a question about the game industry.
#     Returns:
#         List[dict]: Each element contains Platform, Name, YearOfRelease, Description.
#     """
#     # Perform semantic search
#     results = collection.query(
#         query_texts=[query],
#         n_results=15  # Number of results to return
#     )
    
#     games = []
#     for metadata in results['metadatas'][0]:
#         games.append({
#             "Platform": metadata.get("Platform"),
#             "Name": metadata.get("Name"),
#             "YearOfRelease": metadata.get("YearOfRelease"),
#             "Description": metadata.get("Description")
#         })
#     return games
@tool
def retrieve_game(query: str):
    """
    Semantic + keyword search for games.
    """
    query_clean = query.lower().strip()

    # 🔹 Semantic search
    semantic_results = collection.query(
        query_texts=[query_clean],
        n_results=10
    )

    semantic_games = semantic_results['metadatas'][0]

    # 🔹 Keyword filtering boost
    keyword_games = []
    for game in semantic_games:
        text = f"{game.get('Name', '')} {game.get('Description', '')}".lower()
        
        if any(word in text for word in query_clean.split()):
            keyword_games.append(game)

    # 🔹 Merge (keyword hits first)
    final_games = keyword_games + [
        g for g in semantic_games if g not in keyword_games
    ]

    # 🔹 Format output
    return [
        {
            "Platform": g.get("Platform"),
            "Name": g.get("Name"),
            "YearOfRelease": g.get("YearOfRelease"),
            "Description": g.get("Description")
        }
        for g in final_games[:10]
    ]

In [12]:
results = retrieve_game("give info on Pokémon Gold and Silver")
print(results)

# print(collection.count())
# print(collection.peek())

[{'Platform': 'Game Boy Color', 'Name': 'Pokémon Gold and Silver', 'YearOfRelease': 1999, 'Description': 'Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.'}, {'Platform': 'Game Boy Advance', 'Name': 'Pokémon Ruby and Sapphire', 'YearOfRelease': 2002, 'Description': 'Third-generation Pokémon games set in the Hoenn region, featuring new Pokémon and double battles.'}, {'Platform': 'Wii', 'Name': 'Wii Sports', 'YearOfRelease': 2006, 'Description': "A collection of sports games that utilize the Wii's motion controls, bundled with the console to showcase its capabilities."}, {'Platform': 'Super Nintendo Entertainment System (SNES)', 'Name': 'Super Mario World', 'YearOfRelease': 1990, 'Description': 'A classic platformer where Mario embarks on a quest to save Princess Toadstool and Dinosaur Land from Bowser.'}, {'Platform': 'PlayStation 1', 'Name': 'Gran Turismo', 'YearOfRelease': 1997, 'Description': 'A realistic racing simulator featuring a wide arra

#### Evaluate Retrieval Tool

In [13]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

import ollama
import json


@tool
def evaluate_retrieval_tool(question, document):
# prompts can be improved further
    prompt = f"""
You are evaluating whether a retrieved document is useful to answer a user's question.
Consider developer and publisher as same thing.

Question:
{question}

Retrieved document:
{document}

Respond ONLY in JSON format:

{{
 "useful": true or false,
 "reason": "give reason if loaded document is relevant"
}}
"""

    response = ollama.chat(
        model="mistral",
        messages=[{"role": "user", "content": prompt}]
    )

    content = response["message"]["content"]

    try:
        return json.loads(content)
    except:
        return {"useful": False, "reason": "Could not parse response"}



In [44]:
# Example question
question = "when was Pokémon Gold and Silver released?"

# Example document (use one from retrieve_game)
retrieved = retrieve_game(question)
document = retrieved[0]  # Use the first result

# Test the tool
result = evaluate_retrieval_tool(question, document)
print(result)

{'useful': True, 'reason': "The retrieved document provides the year of release for 'Pokémon Gold and Silver' which directly answers the user's question."}


#### Game Web Search Tool

In [14]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

from tavily import TavilyClient
import os

tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

@tool
def web_search_tool(query):
    """
    Tool: Perform web search when internal knowledge is insufficient
    """

    results = tavily.search(
        query=query,
        max_results=3
    )

    return results

In [ ]:
web_search_tool("When was counter strike released?")

{'query': 'When was counter strike released?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.quora.com/When-did-Counter-Strike-come-out',
   'title': 'When did Counter-Strike come out?',
   'content': 'Counter Strike: Source (the full game) was release on November 1, 2004. The beta, however, was released on August 11, 2004, but only to members',
   'score': 0.99939764,
   'raw_content': None},
  {'url': 'https://tradeit.gg/blog/counter-strike-release-date-for-every-game-in-the-series/?srsltid=AfmBOopCjNAam092AyHQDEOh77OMNfLWp2PK0F4AIeYz_swrDTJEacU6',
   'title': 'Counter-Strike Release Date for Every Game in the Series',
   'content': '# Counter-Strike Release Date for Every Game in the Series. We’re about to take a tactical breach into the release dates of every game in the Counter-Strike series. * Counter-Strike started as a Half-Life mod and quickly evolved into a hit series with several notable games including the latest Counter-Str

### Agent

In [15]:
def generate_answer_response(question: str, data: any, source: str) -> str:
    """Generate answer from data"""
    if source == "internal database":
        games = data['games']
        answer = f"Based on our internal database:\n\n"
        for game in games[:2]:
            answer += f"- {game['Name']} was released in {game['YearOfRelease']} for {game['Platform']}\n"
            #answer += f"  Genre: {game['Genre']}, Publisher: {game['Publisher']}\n"
            answer += f"  {game['Description']}\n\n"
        answer += "Source: Internal Game Database"
    else:
        answer = f"Based on web search:\n\n"
        for item in data[:2]:
            answer += f"- {item['title']}\n"
            answer += f"  {item['content'][:200]}...\n"
            answer += f"  Source: {item['url']}\n\n"
    
    return answer

In [17]:
# Create the agent using the course framework and state-machine approach
agent = Agent(
    model_name="gpt-4o-mini",
    instructions="""
    You are UdaPlay, an AI Research Agent for the video game industry. Your goal is to answer questions about video games.

    You have access to three tools:
    1. retrieve_game: Search the internal game database for relevant information
    2. evaluate_retrieval_tool: Evaluate if retrieved information is sufficient to answer the question
    3. web_search_tool: Search the web when internal information is insufficient

    Follow this workflow:
    1. First, try to retrieve relevant information using retrieve_game
    2. Evaluate the retrieved information using evaluate_retrieval_tool
    3. If the evaluation shows the information is not useful, use web_search_tool
    4. Provide a clear, structured answer with sources

    Always be helpful and provide accurate information about video games.
    """,
    tools=[retrieve_game, evaluate_retrieval_tool, web_search_tool]
)

print("✓ UdaPlay Agent ready")

✓ UdaPlay Agent ready


In [19]:
result1 = agent.invoke("Was Mortal Kombat X released for Playstation 5?", session_id="udaplay_demo")
print("Result:", result1)
final_state = result1.get_final_state()
print(final_state["messages"][-1].content)


[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Result: Run('3ca2191b-ee0b-4c0e-8e4e-f3e525f456bf')
Mortal Kombat X was not released for the PlayStation 5. It was originally launched on April 14, 2015, for PlayStation 4, PlayStation 3, Xbox One, Xbox 360, and PC. An upgraded version called Mortal Kombat XL, which included additional content, was released later on March 1, 2016, but it was also only available for PlayStation 4 and Xbox One. 

As of now, Mortal Kombat X has not been specifically released for the PlayStation 5, although it may be playable on the system through backward compatibility with the PlayStation 4 version.

For more detailed information, you can check the [Wikipedia pag

In [22]:
result2 = agent.invoke("When was Marvel's Spider-Man 2 released?", session_id="udaplay_demo")
print("Result:", result2)
final_state = result2.get_final_state()
print(final_state["messages"][-1].content)


[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Result: Run('c3119f72-e822-49ae-8dc9-4bcdf931f2b6')
Marvel's Spider-Man 2 was released on October 20, 2023, for the PlayStation 5. This highly anticipated sequel continues the story of both Peter Parker and Miles Morales as playable characters.


In [18]:
result3 = agent.invoke("When was Pokémon Gold and silver released", session_id="udaplay_demo")
print("Result:", result3)
final_state = result3.get_final_state()
print(final_state["messages"][-1].content)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Result: Run('5a7b4adc-cf14-4786-8df4-1f6a7cdc97a2')
Pokémon Gold and Silver were released in 1999 for the Game Boy Color. These games are part of the second generation of Pokémon and introduced new regions, Pokémon, and gameplay mechanics.


In [29]:
result4 = agent.invoke("When was Brian lara 99 released", session_id="user2")
print("Result:", result4)
final_state = result4.get_final_state()
print(final_state["messages"][-1].content)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Result: Run('6ffe7233-7a4d-4cd0-996f-1519fd2be6c5')
"Brian Lara 99," officially known as "Brian Lara Cricket '99," was released on **December 8, 1998**, for PlayStation. The game was later made available for PC in **1999**.

Here are the key details:
- **Release Date**: December 8, 1998 (PlayStation), 1999 (PC)
- **Developer/Publisher**: Codemasters
- **Genre**: Sports (Cricket)

If you need more information, feel free to ask!


In [30]:
result5 = agent.invoke("When was Halo Infinite released", session_id="user2")
final_state = result5.get_final_state()
print(final_state["messages"][-1].content)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
**Halo Infinite** was released on **December 8, 2021**. It is the latest installment in the Halo franchise, featuring Master Chief's adventures in a new open-world environment.

If you have any more questions about Halo Infinite or other games, feel free to ask!


In [27]:
runs = agent.get_session_runs("udaplay_demo")
for run in runs:
    state = run.get_final_state()
    print("---- RUN ----")
    for msg in state["messages"]:
        print(msg.content)

---- RUN ----

    You are UdaPlay, an AI Research Agent for the video game industry. Your goal is to answer questions about video games.

    You have access to three tools:
    1. retrieve_game: Search the internal game database for relevant information
    2. evaluate_retrieval_tool: Evaluate if retrieved information is sufficient to answer the question
    3. web_search_tool: Search the web when internal information is insufficient

    Follow this workflow:
    1. First, try to retrieve relevant information using retrieve_game
    2. Evaluate the retrieved information using evaluate_retrieval_tool
    3. If the evaluation shows the information is not useful, use web_search_tool
    4. Provide a clear, structured answer with sources

    Always be helpful and provide accurate information about video games.
    
When was Pokémon Gold and silver released
None
"[{'Platform': 'Game Boy Color', 'Name': 'Pok\u00e9mon Gold and Silver', 'YearOfRelease': 1999, 'Description': 'Second-generat

In [31]:
runs = agent.get_session_runs("user2")
for run in runs:
    state = run.get_final_state()
    print("---- RUN ----")
    for msg in state["messages"]:
        print(msg.content)

---- RUN ----

    You are UdaPlay, an AI Research Agent for the video game industry. Your goal is to answer questions about video games.

    You have access to three tools:
    1. retrieve_game: Search the internal game database for relevant information
    2. evaluate_retrieval_tool: Evaluate if retrieved information is sufficient to answer the question
    3. web_search_tool: Search the web when internal information is insufficient

    Follow this workflow:
    1. First, try to retrieve relevant information using retrieve_game
    2. Evaluate the retrieved information using evaluate_retrieval_tool
    3. If the evaluation shows the information is not useful, use web_search_tool
    4. Provide a clear, structured answer with sources

    Always be helpful and provide accurate information about video games.
    
When was Brian lara 99 released
None
"[{'Platform': 'Xbox Series X|S', 'Name': 'Halo Infinite', 'YearOfRelease': 2021, 'Description': \"The latest installment in the Halo fr

### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes